# LTX-Video 0.9.8 distilled **13B** — влезает, но какой ценой

Проверка второй строки из таблицы моделей в `docs/design.md`: у 2B запас
двукратный (G/L ≈ 0,50), значит можно попробовать поднять качество.

**Упирается всё в память, а не в скорость,** причём сразу в двух местах.

Видеопамять, bf16: трансформер ~26 ГБ + T5-XXL ~9,5 ГБ + VAE ~2,5 ГБ = **~38 ГБ**
против 32 на карте. Оперативной памяти на машине **30 ГБ**, поэтому собрать пайплайн
на CPU и потом перенести на карту тоже нельзя — один трансформер займёт 26 ГБ.

Рабочая схема:

1. поднимаем **только T5** на CPU, кодируем промпты, сразу выгружаем;
2. грузим трансформер **сразу на GPU** через `device_map="cuda"`, минуя оперативку;
3. собираем пайплайн **без текстового энкодера** — `encode_prompt` трогает T5
   только когда `prompt_embeds is None`, а мы всегда передаём готовые эмбеддинги;
4. `expandable_segments` против фрагментации и тайлинг VAE — без них не хватает
   даже на активации.

Для стрима схема с заранее подготовленными эмбеддингами и так правильная: промпт
меняется редко, а кодировать его можно, пока считается текущий кусок.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # до импорта torch!

!pip install -q -U diffusers transformers accelerate
!pip install -q tiktoken sentencepiece protobuf imageio imageio-ffmpeg

In [ ]:
import torch
print(torch.__version__, torch.version.cuda)
free, total = torch.cuda.mem_get_info()
print(f"{torch.cuda.get_device_name(0)}: свободно {free/1e9:.1f} из {total/1e9:.1f} ГБ")
assert free / 1e9 > 30, "нужна пустая карта: 13B занимает 29 ГБ только весами" 

## Шаг 1: кодируем промпты и убираем T5

Пайплайн собирается «наполовину» — только с текстовым энкодером, остальное `None`.
`LTXConditionPipeline` это допускает: в `__init__` все обращения к компонентам идут
через `getattr(..., None)`.

T5 берём из `Lightricks/LTX-Video` — он уже в кэше после 2B, качать 19 ГБ заново не нужно.
На CPU кодирование занимает пару минут.

In [ ]:
import time, gc
from diffusers import (LTXConditionPipeline, LTXVideoTransformer3DModel,
                       AutoencoderKLLTXVideo, FlowMatchEulerDiscreteScheduler)
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from transformers import T5EncoderModel, T5Tokenizer
from diffusers.utils import load_image, export_to_video

REPO, T5_REPO = "Lightricks/LTX-Video-0.9.8-13B-distilled", "Lightricks/LTX-Video"
DT, FPS = torch.bfloat16, 24

PROMPTS = {
    "base": ("A man with short gray hair plays a red electric guitar on a small stage, "
             "warm stage lights, smooth camera movement"),
    "blue": "The same man on stage, the lights turn deep blue, fog rolls across the floor",
}

t0 = time.time()
text_encoder = T5EncoderModel.from_pretrained(T5_REPO, subfolder="text_encoder", dtype=DT)
tokenizer = T5Tokenizer.from_pretrained(T5_REPO, subfolder="tokenizer")
encoder_only = LTXConditionPipeline(transformer=None, vae=None, scheduler=None,
                                    text_encoder=text_encoder, tokenizer=tokenizer)
embeds = {}
with torch.no_grad():
    for key, text in PROMPTS.items():
        emb, mask, _, _ = encoder_only.encode_prompt(
            prompt=text, do_classifier_free_guidance=False,
            device=torch.device("cpu"), dtype=DT)
        embeds[key] = (emb.to("cuda"), mask.to("cuda"))

del encoder_only, text_encoder, tokenizer
gc.collect()
print(f"закодировано {len(embeds)} промптов на CPU за {time.time() - t0:.1f}s, T5 выгружен")

## Шаг 2: трансформер и VAE — на карту

`device_map="cuda"` заставляет accelerate раскладывать шарды прямо в видеопамять,
не собирая полную копию в оперативке.

VAE обязан быть на карте: он нужен не только для декода, но и для кодирования
опорной картинки или хвоста предыдущего куска. Держать его на CPU не выйдет —
получите `Input type (CUDABFloat16Type) and weight type (CPUBFloat16Type)`.

In [ ]:
t0 = time.time()
transformer = LTXVideoTransformer3DModel.from_pretrained(
    REPO, subfolder="transformer", dtype=DT, device_map="cuda")
vae = AutoencoderKLLTXVideo.from_pretrained(REPO, subfolder="vae", dtype=DT).to("cuda")
vae.enable_tiling()        # без тайлинга декод не помещается

pipe = LTXConditionPipeline(
    transformer=transformer, vae=vae,
    scheduler=FlowMatchEulerDiscreteScheduler.from_pretrained(REPO, subfolder="scheduler"),
    text_encoder=None, tokenizer=None)
print(f"загрузка на GPU: {time.time() - t0:.1f}s")

free, total = torch.cuda.mem_get_info()
print(f"занято весами {(total - free) / 1e9:.1f} ГБ, на активации остаётся {free / 1e9:.1f} ГБ")

## Сколько влезает

Свободного места под активации остаётся около 4,5 ГБ, и это решает всё.
Перебираем разрешения и длины куска.

In [ ]:
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
)
cond = LTXVideoCondition(image=image, frame_index=0)

def run(width, height, num_frames, steps=8, conditions=None, seed=0, key="base"):
    emb, mask = embeds[key]
    return pipe(conditions=conditions or [cond],
                prompt_embeds=emb, prompt_attention_mask=mask,
                width=width, height=height, num_frames=num_frames,
                num_inference_steps=steps, guidance_scale=1.0,
                generator=torch.Generator("cuda").manual_seed(seed)).frames[0]

def probe(width, height, num_frames, runs=2):
    label = f"{width}x{height}, {num_frames} кадров"
    try:
        for i in range(runs):                      # первый прогон прогревочный
            torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            frames = run(width, height, num_frames, seed=i)
            G = time.time() - t0
        L = num_frames / FPS
        print(f"  влезло  {label}: G={G:.2f}s  L={L:.2f}s  G/L={G / L:.2f}  "
              f"пик {torch.cuda.max_memory_allocated() / 1e9:.1f} ГБ")
        return frames
    except torch.OutOfMemoryError:
        print(f"  OOM     {label}")
    finally:
        gc.collect(); torch.cuda.empty_cache()

for cfg in [(704, 480, 97), (768, 512, 97), (704, 480, 49), (512, 320, 97)]:
    frames = probe(*cfg)

Замерено на пустой 5090 (всего на карте 33,7 ГБ по счёту `mem_get_info`):

| конфигурация | G | G/L | пик VRAM |
|---|---|---|---|
| 704×480, 97 кадров | 7,05 с | **1,75** | 31,0 ГБ |
| 768×512, 97 кадров | 8,65 с | **2,14** | 31,2 ГБ |
| 704×480, 49 кадров | 3,27 с | **1,60** | 30,5 ГБ |
| 512×320, 97 кадров | 2,94 с | **0,73** | 30,2 ГБ |

Разброс между прогонами — около ±10%, так что сравнивать стоит порядок величины,
а не третий знак.

Единственная конфигурация с G/L < 1 — **512×320**. На разрешении, где 2B давала
G/L = 0,50, тринадцатимиллиардная отстаёт от реального времени в полтора раза.

Короче кусок не помогает: G падает пропорционально, отношение не меняется.

## Цепочка на единственной рабочей конфигурации

In [ ]:
import numpy as np

def generate_chain_13b(start_image, key_for, n_chunks, chunk_frames=97, overlap=9,
                       width=512, height=320, steps=8, seed=0, fps=FPS):
    for name, v in (("chunk_frames", chunk_frames), ("overlap", overlap)):
        if (v - 1) % 8 != 0:
            raise ValueError(f"{name} должен быть вида 8k+1, получено {v}")

    frames, stats, tail = [], [], None
    for i in range(n_chunks):
        c = (LTXVideoCondition(image=start_image, frame_index=0) if tail is None
             else LTXVideoCondition(video=tail, frame_index=0))
        t0 = time.time()
        out = run(width, height, chunk_frames, steps=steps,
                  conditions=[c], seed=seed + i, key=key_for(i))
        G = time.time() - t0

        new = out if tail is None else out[overlap:]
        frames.extend(new)
        tail = out[-overlap:]

        L = len(new) / fps
        stats.append({"G": G, "L": L, "ratio": G / L})
        print(f"кусок {i:3d}  G={G:5.2f}s  L={L:4.2f}s  G/L={G / L:4.2f}  "
              f"всего {len(frames) / fps:6.1f}s", flush=True)
    return frames, stats

torch.cuda.reset_peak_memory_stats()
frames, stats = generate_chain_13b(image, lambda i: "base", n_chunks=8)
export_to_video(frames, "chain_13b.mp4", fps=FPS)

r = np.array([s["ratio"] for s in stats[1:]])
G = np.array([s["G"] for s in stats[1:]])
print(f"\n13B @512x320: G медиана {np.median(G):.2f}s, G/L медиана {np.median(r):.2f}, худший {r.max():.2f}")
print(f"пик VRAM {torch.cuda.max_memory_allocated() / 1e9:.1f} ГБ")
print("для сравнения 2B @704x480: G 1.83s, G/L 0.50")

In [ ]:
from IPython.display import Video
Video("chain_13b.mp4", embed=True, width=704)

## fp8: снимаем барьер памяти

Родной `ltxv-13b-0.9.8-distilled-fp8.safetensors` для diffusers не годится: в его
заголовке 480 тензоров `F8_E4M3` **без единого масштабного коэффициента**
(`scale_shift_table` — это параметр AdaLN, не квантование). Без масштабов веса либо
разожмутся обратно в bf16, либо нужны свои fp8-ядра, как в родном коде LTX.

Поэтому квантуем сами — torchao поверх уже загруженных bf16-весов. Карта sm120,
fp8-тензорные ядра есть.

`diffusers.TorchAoConfig` здесь не работает (0.40 не дружит с torchao 0.18:
«Unable to import torchao Tensor objects»), но `quantize_` применяется к модели
напрямую и этого достаточно.

In [ ]:
from torchao.quantization import quantize_, Float8DynamicActivationFloat8WeightConfig

t0 = time.time()
quantize_(pipe.transformer, Float8DynamicActivationFloat8WeightConfig())
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"квантование за {time.time() - t0:.1f}s: занято {(total - free) / 1e9:.1f} ГБ, "
      f"свободно {free / 1e9:.1f} ГБ")

for cfg in [(704, 480, 97), (768, 512, 97), (512, 320, 97)]:
    probe(*cfg)

## fp8 + torch.compile

Компиляция под каждое разрешение своя (`dynamic=False`), первый прогон на новой
форме — это минуты, поэтому `probe` с прогревом здесь особенно важен.

In [ ]:
t0 = time.time()
pipe.transformer.compile(mode="default", fullgraph=False, dynamic=False)
print(f"compile навешан за {time.time() - t0:.1f}s (компиляция — на первом прогоне каждой формы)")

for cfg in [(704, 480, 97), (640, 384, 97), (576, 352, 97), (512, 320, 97)]:
    probe(*cfg)

Замерено (один прогон ноутбука, разброс между прогонами ±10%):

| конфигурация | bf16 | fp8 | fp8 + compile |
|---|---|---|---|
| 704×480 | 7,05 с / 1,75 | 6,25 с / 1,55 | **4,79 с / 1,19** |
| 768×512 | 8,65 с / 2,14 | 7,90 с / 1,95 | — |
| 640×384 | — | — | **2,76 с / 0,68** |
| 576×352 | — | — | **2,26 с / 0,56** |
| 512×320 | 2,94 с / 0,73 | 2,81 с / 0,69 | **1,84 с / 0,46** |
| пик VRAM | 30–31 ГБ | **17–18 ГБ** | **17–18 ГБ** |

*(в ячейках G / G/L для одиночного куска; в цепочке с перекрытием отношение выше примерно на 10%)*

Квантование заняло доли секунды и срезало веса трансформера с 26,6 до 14,1 ГБ.
**Барьер памяти снят полностью** — свободно ~17 ГБ вместо 3,5. Артефактов
квантования на картинке не видно.

Скорость fp8 почти не даёт (−10%), основной вклад у `torch.compile` (−25%).
Вместе они ускоряют 13B примерно вдвое, но на 704×480 этого всё равно не хватает.

## Вывод: 13B вытянуть можно, но смысла нет

Граница реального времени для 13B проходит около **640×384**. Сравним с 2B на его
рабочем разрешении — один сид, один промпт, одна опорная картинка:

| | 13B fp8+compile @640×384 | 2B bf16 @704×480 |
|---|---|---|
| G | 2,8–3,1 с | **1,8–2,1 с** |
| G/L | 0,68–0,77 | **0,46–0,51** |
| пик VRAM | 17,5 ГБ | 19,3 ГБ |

2B и быстрее, и **выглядит лучше**: у 13B на 640×384 смазывается лицо (рот, подбородок,
гарнитура вросла в щёку), у 2B читаются лады, звукосниматели и детали фона.
Преимущество большой модели не доживает до разрешения, которое она может себе
позволить по скорости.

**Решение: остаёмся на 2B.** fp8 при этом стоит запомнить — он снимает память
начисто, и если появится 13B вдвое быстрее или карта помощнее, путь уже разведан.
`torch.compile` полезен и для 2B — это следующий шаг там.